In [1]:
pip install langchain langchain-community langchain-groq pypdf faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import  RecursiveCharacterTextSplitter


/tmp/ipykernel_892/1994472649.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [8]:
loader = PyPDFLoader("/content/Lecture 6  - Supervised ML (Classification Part 2 SVM + DT).pdf")
documents =loader.load()
print(f"number of pages: {len(documents)}")

number of pages: 89


In [9]:
text_spiliter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks=text_spiliter.split_documents(documents)
print("number of chunks",len(chunks))

number of chunks 85


In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstores=FAISS.from_documents(chunks,embeddings)
vectorstores.save_local("faiss_index")

/tmp/ipykernel_892/4263957994.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
vectorstores=FAISS.load_local("faiss_index",embeddings,
                               allow_dangerous_deserialization=True
                              )

retriever=vectorstores.as_retriever(search_kwargs={"k":3})





In [ ]:

from groq import Groq
from config import API_KEY


client = Groq(api_key=API_KEY)

In [14]:
pip install langchain-classic

In [15]:
from langchain_classic.chains import RetrievalQA
llm=ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [18]:
query = "What is the main idea in this document?"
result = qa_chain.invoke({"query": query})

print("Answer:", result["result"])
print("\n--- Sources ---")
for doc in result["source_documents"]:
    print(f"Page {doc.metadata.get('page')}: {doc.page_content[:100]}...")

Answer: The main idea in this document appears to be related to Artificial Intelligence (AI) and Machine Learning, specifically introducing concepts such as decision trees and classification processes.

--- Sources ---
Page 27: The decision tree 
structure 
• It is a graphical representation for getting all the 
possible solut...
Page 0: Artificial Intelligence Concepts and 
Machine Learning Techniques
(AI – 103)
Walaa H. Elashmawi
Prof...
Page 34: Classification 
process...


In [20]:
from langchain_core.prompts import PromptTemplate

custom_prompt = PromptTemplate(
    template="""Use only the following context to answer the question. If the answer isn't in the context, say "Not found in the document."

Context: {context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": custom_prompt},
    return_source_documents=True
)

In [21]:
print("Ask any question about the document (type 'exit' to quit)\n")

while True:
    query = input("Your question: ")

    if query.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    result = qa_chain.invoke({"query": query})

    print("\nAnswer:", result["result"])

    print("\n--- Sources ---")
    for doc in result["source_documents"]:
        print(f"Page {doc.metadata.get('page')}: {doc.page_content[:100]}...")

    print("\n" + "-"*50 + "\n")

Ask any question about the document (type 'exit' to quit)

Your question: what is SVM

Answer: SVM is a supervised machine learning algorithm which is mainly used to classify data into different classes.

--- Sources ---
Page 4: What is 
Support Vector 
Machine (SVM)
SVM is a supervised machine learning algorithm 
which is main...
Page 3: Support Vector Machine...
Page 9: Linear SVMs Mathematically (cont.)
• Then we can formulate the quadratic optimization problem: 
• Wh...

--------------------------------------------------

Your question: explain Linear Separators

Answer: Linear separators can be viewed as the task of separating classes in feature space using a classifier defined as f(x) = sign(wTx + b), where w is the decision hyperplane normal vector. The goal is to find the optimal linear separator. The distance from an example xi to the separator is an important concept, and the examples closest to the hyperplane are called support vectors. The margin ρ of the separator is the d

In [22]:
chat_history = []

while True:
    query = input("Your question: ")

    if query.lower() in ["exit", "quit"]:
        break

    result = qa_chain.invoke({"query": query})
    answer = result["result"]

    chat_history.append({"question": query, "answer": answer})

    print("\nAnswer:", answer, "\n")

# Print all questions and answers at the end
for i, item in enumerate(chat_history, 1):
    print(f"{i}. Q: {item['question']}\n   A: {item['answer']}\n")

Your question: what is svm

Answer: SVM is a supervised machine learning algorithm which is mainly used to classify data into different classes. 

Your question: explain Soft Margin

Answer: Soft Margin Classification is used when the training set is not linearly separable. It incorporates slack variables (ξi) to allow for misclassification of difficult or noisy examples. This results in a "soft" margin, which means that some examples can be on the wrong side of the hyperplane. The slack variables (ξi) are added to the constraint equations, allowing for some examples to violate the margin condition. The parameter C controls the trade-off between maximizing the margin and fitting the training data, acting as a regularization term. 

Your question: exit
1. Q: what is svm
   A: SVM is a supervised machine learning algorithm which is mainly used to classify data into different classes.

2. Q: explain Soft Margin
   A: Soft Margin Classification is used when the training set is not linearly